# Advanced RAG: фильтрация, гибридный поиск и Neo4j

На документах РобоТех последовательно сравним dense-поиск, фильтрацию по метаданным, hybrid search и cross-encoder. Затем посмотрим готовый граф в Neo4j. Основа примера: `llamaindex/advanced_rag_llamaindex_demo.py` и `data/knowledge_base/`.

## Подготовка

Из корня репозитория выполните:

```bash
uv sync
docker compose up -d qdrant
docker compose --profile graph up -d neo4j
ollama pull hf.co/Qwen/Qwen3-0.6B-GGUF:Q8_0
```

Выберите ядро `.venv` и запускайте ячейки по порядку. Qdrant нужен для поиска, Neo4j — для графового примера, Ollama — только для генерации и переписывания запроса. При первом запуске модели embeddings, BM25 и cross-encoder могут скачиваться. Графовый раздел можно выполнить отдельно после первой ячейки с импортами.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
pd.set_option("display.max_colwidth", None)
from IPython.display import display
from llama_index.core import Settings, StorageContext, VectorStoreIndex
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.core.prompts import PromptTemplate
from llama_index.core.schema import NodeWithScore
from llama_index.core.vector_stores.types import VectorStoreQueryMode

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "llamaindex" / "advanced_rag_llamaindex_demo.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Откройте ноутбук внутри репозитория rag-engineering-workshop")
sys.path.insert(0, str(ROOT / "llamaindex"))
import advanced_rag_llamaindex_demo as demo

MODEL = "hf.co/Qwen/Qwen3-0.6B-GGUF:Q8_0"
COLLECTION = "robotex_advanced_notebook_hybrid"
print("Репозиторий:", ROOT)
print("Python:", sys.executable)

Репозиторий: /Users/newuser/Documents/repo/rag-engineering-workshop
Python: /Users/newuser/Documents/repo/rag-engineering-workshop/.venv/bin/python


## 1. Документы и метаданные

Загрузчик проекта читает Markdown и переносит метаданные в `Document`. Если год или категория не заданы в файле, загрузчик пытается вывести их из имени и текста; это не проверка актуальности.

In [2]:
documents = demo.load_documents()
print("Документов:", len(documents))
display(pd.DataFrame([{
    "Файл": doc.metadata.get("filename"),
    "Год": doc.metadata.get("year"),
    "Категория": doc.metadata.get("category"),
} for doc in documents]).head(10))

# Посмотрим, как текст документа превращается в nodes.
splitter = demo.SentenceSplitter(chunk_size=demo.CHUNK_SIZE, chunk_overlap=demo.CHUNK_OVERLAP)
nodes = splitter.get_nodes_from_documents(documents)
print("Фрагментов:", len(nodes))
print("Пример метаданных:", nodes[0].metadata)
print(nodes[0].get_content(metadata_mode="none"))

Документов: 36


,Файл,Год,Категория
0,Doc_Filler_1.md,2026,General
1,Doc_Filler_10.md,2026,General
2,Doc_Filler_11.md,2026,General
3,Doc_Filler_12.md,2026,General
4,Doc_Filler_13.md,2026,General
5,Doc_Filler_14.md,2026,General
6,Doc_Filler_15.md,2026,General
7,Doc_Filler_16.md,2026,General
8,Doc_Filler_17.md,2026,General
9,Doc_Filler_18.md,2026,General


Фрагментов: 38
Пример метаданных: {'source': 'data/knowledge_base/Doc_Filler_1.md', 'filename': 'Doc_Filler_1.md', 'year': 2026, 'category': 'General'}
# Инструкция по Настройке Integrated Development Environment (IDE) JetBrains для РобоТех

1. **Скачивание и Установка**
    Сначала необходимо скачать IDE JetBrains с официального сайта [jetbrains.com](https://www.jetbrains.com/). После загрузки установите приложение на свой компьютер в соответствии с инструкциями для вашей операционной системы.

2. **Настройка IDE**
    Вход в IDE JetBrains осуществляется путем авторизации с помощью учётной записи JetBrains или е-mail и пароля. После успешного входа рекомендуется настроить необходимые настройки для проекта РобоТех:
   - **Настройка языковой среды** — выберите `Ruby` (для Ruby-проектов) или другой подходящий язык в зависимости от вашего проекта.
   - **Настройка плагинов** — инсталлируйте необходимые плагины для улучшения эффективности работы с IDE JetBrains (например, Rubocop для автом

## 2. Общий индекс в Qdrant

LlamaIndex разбивает документы на фрагменты. Для каждого фрагмента Qdrant хранит dense-вектор BGE-M3 и sparse-вектор BM25. Если коллекция уже существует, используем её повторно: после изменения файлов она может содержать старые данные. Для обновления удалите только коллекцию `robotex_advanced_notebook_hybrid` через интерфейс Qdrant и повторите ячейку.

In [3]:
if not demo.check_qdrant():
    raise RuntimeError("Запустите Qdrant: docker compose up -d qdrant")

demo.configure_settings(MODEL, enable_tracing=False)
if demo.collection_exists(COLLECTION):
    index = demo.load_hybrid_index(collection_name=COLLECTION, model=MODEL, enable_tracing=False)
    print("Используем существующую коллекцию:", COLLECTION)
else:
    storage = StorageContext.from_defaults(vector_store=demo.hybrid_vector_store(COLLECTION))
    index = VectorStoreIndex.from_documents(documents, storage_context=storage, show_progress=True)
    print("Создана коллекция:", COLLECTION)

Проверка Qdrant... OK, коллекций: 1
Версия Qdrant: 1.16.2
Embedding model: BAAI/bge-m3 on mps
LLM via LlamaIndex: hf.co/Qwen/Qwen3-0.6B-GGUF:Q8_0 at http://localhost:11434
LLM request timeout: 600s


Embedding model: BAAI/bge-m3 on mps
LLM via LlamaIndex: hf.co/Qwen/Qwen3-0.6B-GGUF:Q8_0 at http://localhost:11434
LLM request timeout: 600s


Используем существующую коллекцию: robotex_advanced_notebook_hybrid


## 3. Dense-поиск и фильтр

Сначала ищем без ограничений, затем оставляем документы с `year=2026`. Фильтр ограничивает выдачу по указанному полю, но не определяет, действует ли документ.

In [4]:
def show_results(items):
    display(pd.DataFrame([{
        "Ранг": i,
        "Источник": demo.source_filename(item),
        "Год": item.node.metadata.get("year"),
        "Score": item.score,
    } for i, item in enumerate(items, 1)]))
    for i, item in enumerate(items, 1):
        print(f"\n--- Фрагмент {i}: {demo.source_filename(item)} ---")
        print(item.node.get_content(metadata_mode="none"))

question = demo.LECTURE_RERANK_QUESTION
print("Вопрос:", question)
dense = demo.retrieve_nodes(query=question, index=index, top_k=3)
show_results(dense)

Вопрос: Можно ли пользоваться VPN для обычной ежедневной работы, а не только при аварии?


,Ранг,Источник,Год,Score
0,1,Doc_Filler_7.md,2026,0.558556
1,2,Doc_Filler_19.md,2026,0.553605
2,3,Doc_Filler_5.md,2026,0.473398



--- Фрагмент 1: Doc_Filler_7.md ---
**Руководство по использованию VPN-клиента в компании РобоТех**

1. **Цель и назначение**

Инструкция предназначена для описания процедуры установки и использования Virtual Private Network (VPN) клиента, обеспечивающего защищенное соединение с внешней сетью компании РобоТех.

2. **Установка VPN-клиента**

Инсталляция VPN-клиента может быть выполнена следующими способами:
- Скачивание и установка утилиты по ссылке, предоставленной администратором сети.
- Установка VPN-клиента при помощи управляющей системы (Windows Update или App Store).

3. **Настройка VPN-клиента**

После установки необходимо выполнить настройку VPN-клиента:
- Зарегистрироваться в компании РобоТех, если это не сделано ранее.
- Выбрать сервер расположенный в сети компании РобоТех.
- Установить безопасное соединение и автоматическое подключение к серверу.

4. **Использование VPN-клиента**

После успешной настройки VPN-клиент будет автоматически подключаться при включении компьютера и

In [5]:
filtered = demo.retrieve_nodes(query=question, index=index, top_k=3, year=2026)
show_results(filtered)

,Ранг,Источник,Год,Score
0,1,Doc_Filler_7.md,2026,0.558556
1,2,Doc_Filler_19.md,2026,0.553605
2,3,Doc_Filler_5.md,2026,0.473398



--- Фрагмент 1: Doc_Filler_7.md ---
**Руководство по использованию VPN-клиента в компании РобоТех**

1. **Цель и назначение**

Инструкция предназначена для описания процедуры установки и использования Virtual Private Network (VPN) клиента, обеспечивающего защищенное соединение с внешней сетью компании РобоТех.

2. **Установка VPN-клиента**

Инсталляция VPN-клиента может быть выполнена следующими способами:
- Скачивание и установка утилиты по ссылке, предоставленной администратором сети.
- Установка VPN-клиента при помощи управляющей системы (Windows Update или App Store).

3. **Настройка VPN-клиента**

После установки необходимо выполнить настройку VPN-клиента:
- Зарегистрироваться в компании РобоТех, если это не сделано ранее.
- Выбрать сервер расположенный в сети компании РобоТех.
- Установить безопасное соединение и автоматическое подключение к серверу.

4. **Использование VPN-клиента**

После успешной настройки VPN-клиент будет автоматически подключаться при включении компьютера и

## 4. Hybrid search

Объединяем семантический dense-поиск с BM25. Второй сигнал помогает при поиске слов и кодов; результат зависит от запроса и данных. Используем тот же фильтр по году.

In [6]:
hybrid = demo.retrieve_nodes(query=question, index=index, top_k=8, year=2026, hybrid=True)
show_results(hybrid[:3])

,Ранг,Источник,Год,Score
0,1,Doc_Filler_7.md,2026,1.000000
1,2,Doc_Filler_19.md,2026,0.776903
2,3,Doc_Filler_14.md,2026,0.309179



--- Фрагмент 1: Doc_Filler_7.md ---
**Руководство по использованию VPN-клиента в компании РобоТех**

1. **Цель и назначение**

Инструкция предназначена для описания процедуры установки и использования Virtual Private Network (VPN) клиента, обеспечивающего защищенное соединение с внешней сетью компании РобоТех.

2. **Установка VPN-клиента**

Инсталляция VPN-клиента может быть выполнена следующими способами:
- Скачивание и установка утилиты по ссылке, предоставленной администратором сети.
- Установка VPN-клиента при помощи управляющей системы (Windows Update или App Store).

3. **Настройка VPN-клиента**

После установки необходимо выполнить настройку VPN-клиента:
- Зарегистрироваться в компании РобоТех, если это не сделано ранее.
- Выбрать сервер расположенный в сети компании РобоТех.
- Установить безопасное соединение и автоматическое подключение к серверу.

4. **Использование VPN-клиента**

После успешной настройки VPN-клиент будет автоматически подключаться при включении компьютера и

## 5. Cross-encoder reranking

Модель оценивает пары «вопрос + фрагмент» и пересортировывает восемь кандидатов. Если нужный фрагмент не попал в кандидаты, reranker его не добавит. Его оценки нельзя напрямую сравнивать с оценками retrieval.

In [7]:
reranker = SentenceTransformerRerank(
    model=demo.RERANK_MODEL, top_n=3, device=demo.select_device(),
    keep_retrieval_score=False,
)
# Создаём новые обёртки: reranker меняет score.
reranked = reranker.postprocess_nodes(
    [NodeWithScore(node=item.node, score=item.score) for item in hybrid],
    query_str=question,
)
print("До reranking:")
show_results(hybrid)
print("После reranking:")
show_results(reranked)

До reranking:


,Ранг,Источник,Год,Score
0,1,Doc_Filler_7.md,2026,1.000000
1,2,Doc_Filler_19.md,2026,0.776903
2,3,Doc_Filler_14.md,2026,0.309179
3,4,Doc_Filler_18.md,2026,0.274833
4,5,IT_Zero_Trust_2026.md,2026,0.270744
5,6,Doc_Filler_5.md,2026,0.145874
6,7,Security_Passwords.md,2026,0.084356
7,8,Doc_Filler_17.md,2026,0.081948



--- Фрагмент 1: Doc_Filler_7.md ---
**Руководство по использованию VPN-клиента в компании РобоТех**

1. **Цель и назначение**

Инструкция предназначена для описания процедуры установки и использования Virtual Private Network (VPN) клиента, обеспечивающего защищенное соединение с внешней сетью компании РобоТех.

2. **Установка VPN-клиента**

Инсталляция VPN-клиента может быть выполнена следующими способами:
- Скачивание и установка утилиты по ссылке, предоставленной администратором сети.
- Установка VPN-клиента при помощи управляющей системы (Windows Update или App Store).

3. **Настройка VPN-клиента**

После установки необходимо выполнить настройку VPN-клиента:
- Зарегистрироваться в компании РобоТех, если это не сделано ранее.
- Выбрать сервер расположенный в сети компании РобоТех.
- Установить безопасное соединение и автоматическое подключение к серверу.

4. **Использование VPN-клиента**

После успешной настройки VPN-клиент будет автоматически подключаться при включении компьютера и

,Ранг,Источник,Год,Score
0,1,IT_Zero_Trust_2026.md,2026,-0.528834
1,2,Doc_Filler_7.md,2026,-1.879383
2,3,Doc_Filler_14.md,2026,-4.483132



--- Фрагмент 1: IT_Zero_Trust_2026.md ---
# Стандарт доступа Zero Trust 2026

С 2026 года все инженерные сервисы РобоТех подключаются через Zero Trust Access. Внутренний код стандарта: ZTA-17.

Для доступа к Qdrant, Neo4j, GitHub Actions и окружениям customer-data требуется корпоративный SSO, аппаратный ключ FIDO2 и активная проверка устройства.

VPN остается резервным каналом только для аварийного доступа. Постоянное использование VPN вместо Zero Trust запрещено.

Иван Петров отвечает за внедрение ZTA-17 в команде Platform. Команда Security проводит ежеквартальный аудит доступа.

Связанные системы: Qdrant, Neo4j, Ollama, GitHub Actions.

--- Фрагмент 2: Doc_Filler_7.md ---
**Руководство по использованию VPN-клиента в компании РобоТех**

1. **Цель и назначение**

Инструкция предназначена для описания процедуры установки и использования Virtual Private Network (VPN) клиента, обеспечивающего защищенное соединение с внешней сетью компании РобоТех.

2. **Установка VPN-клиента**

Инсталляц

## 6. Ответ по найденным фрагментам

Query engine повторяет hybrid retrieval, применяет тот же reranker и передаёт результаты в Ollama. Проверьте ответ по показанным источникам: модель может ошибаться.

In [8]:
if not demo.check_ollama(MODEL):
    raise RuntimeError(f"Запустите Ollama и скачайте модель: ollama pull {MODEL}")

query_engine = index.as_query_engine(
    similarity_top_k=8, sparse_top_k=12, hybrid_top_k=8, alpha=0.5,
    vector_store_query_mode=VectorStoreQueryMode.HYBRID,
    filters=demo.metadata_filters(year=2026),
    node_postprocessors=[reranker],
    text_qa_template=PromptTemplate(demo.QA_TEMPLATE),
)
response = query_engine.query(question)
print(str(response))
show_results(response.source_nodes)

Проверка Ollama... OK
Проверка модели hf.co/Qwen/Qwen3-0.6B-GGUF:Q8_0... найдена


В документах нет информации об этом.


,Ранг,Источник,Год,Score
0,1,IT_Zero_Trust_2026.md,2026,-0.528834
1,2,Doc_Filler_7.md,2026,-1.879383
2,3,HR_policy_2026.md,2026,-3.769705



--- Фрагмент 1: IT_Zero_Trust_2026.md ---
# Стандарт доступа Zero Trust 2026

С 2026 года все инженерные сервисы РобоТех подключаются через Zero Trust Access. Внутренний код стандарта: ZTA-17.

Для доступа к Qdrant, Neo4j, GitHub Actions и окружениям customer-data требуется корпоративный SSO, аппаратный ключ FIDO2 и активная проверка устройства.

VPN остается резервным каналом только для аварийного доступа. Постоянное использование VPN вместо Zero Trust запрещено.

Иван Петров отвечает за внедрение ZTA-17 в команде Platform. Команда Security проводит ежеквартальный аудит доступа.

Связанные системы: Qdrant, Neo4j, Ollama, GitHub Actions.

--- Фрагмент 2: Doc_Filler_7.md ---
**Руководство по использованию VPN-клиента в компании РобоТех**

1. **Цель и назначение**

Инструкция предназначена для описания процедуры установки и использования Virtual Private Network (VPN) клиента, обеспечивающего защищенное соединение с внешней сетью компании РобоТех.

2. **Установка VPN-клиента**

Инсталляц

## 7. Контекстный вопрос: query rewriting

В отдельном примере из репозитория короткий вопрос уточняется с помощью истории. Сравним результаты до и после переписывания. Модель может изменить смысл, поэтому проверяем полученный запрос.

In [9]:
history = "До этого мы обсуждали стандарт доступа ZTA-17 для инженерных сервисов."
short_question = "А кто за это отвечает?"
rewritten = demo.rewrite_query(short_question, history, model=MODEL)
print("История:", history)
print("Исходный вопрос:", short_question)
print("Уточнённый запрос:", rewritten)
print("До переписывания:")
show_results(demo.retrieve_nodes(query=short_question, index=index, top_k=3, year=2026, hybrid=True))
print("После переписывания:")
show_results(demo.retrieve_nodes(query=rewritten, index=index, top_k=3, year=2026, hybrid=True))

История: До этого мы обсуждали стандарт доступа ZTA-17 для инженерных сервисов.
Исходный вопрос: А кто за это отвечает?
Уточнённый запрос: Кто в истории отвечает за доступ к системе ZTA-17 для инженерных сервисов?
До переписывания:


,Ранг,Источник,Год,Score
0,1,Meeting_Atlas_Security_2026.md,2026,0.892271
1,2,Doc_Filler_18.md,2026,0.500000
2,3,Project_Atlas_Graph_2026.md,2026,0.313307



--- Фрагмент 1: Meeting_Atlas_Security_2026.md ---
# Встреча Atlas и Security по графовому RAG

Дата: 2026-03-14.

Участники: Иван Петров, Мария Соколова, Анна Ким, команда Security, команда AI Platform.

Решения:
- Для графового поиска используется Neo4j PropertyGraphIndex.
- Для точных кодов политик используется hybrid search с sparse-сигналом BM25.
- Для финального порядка источников используется Cross-Encoder Reranking.
- Для вопросов про текущие HR-правила обязателен metadata filter year=2026 и category=HR.

Иван Петров должен подготовить демонстрацию Graph Retrieval для вопроса: "Кто отвечает за проект Atlas и какие системы он использует?"

Мария Соколова должна проверить корректность ответа по политике HR-REMOTE-2026.

--- Фрагмент 2: Doc_Filler_18.md ---
В данном документе я применил жирный текст для выделения заголовков и важных терминов, которые могут быть полезны для русскоговорящего аудитории.
3. Для лучшего понимания некоторых правил может понадобиться дополнительное объя

,Ранг,Источник,Год,Score
0,1,IT_Zero_Trust_2026.md,2026,0.967713
1,2,Project_Atlas_Graph_2026.md,2026,0.500000
2,3,Meeting_Atlas_Security_2026.md,2026,0.129366



--- Фрагмент 1: IT_Zero_Trust_2026.md ---
# Стандарт доступа Zero Trust 2026

С 2026 года все инженерные сервисы РобоТех подключаются через Zero Trust Access. Внутренний код стандарта: ZTA-17.

Для доступа к Qdrant, Neo4j, GitHub Actions и окружениям customer-data требуется корпоративный SSO, аппаратный ключ FIDO2 и активная проверка устройства.

VPN остается резервным каналом только для аварийного доступа. Постоянное использование VPN вместо Zero Trust запрещено.

Иван Петров отвечает за внедрение ZTA-17 в команде Platform. Команда Security проводит ежеквартальный аудит доступа.

Связанные системы: Qdrant, Neo4j, Ollama, GitHub Actions.

--- Фрагмент 2: Project_Atlas_Graph_2026.md ---
# Проект Atlas: графовая навигация по корпоративным знаниям

Проект Atlas строит Graph RAG для корпоративных документов РобоТех. Цель проекта: связать документы, сотрудников, команды, политики и технические системы в одном property graph.

Мария Соколова является владельцем бизнес-требований проекта Atl

## 8. Графовый поиск в существующем Neo4j

В проекте граф строится функцией `seed_manual_graph` через Python driver. В нём есть документы, фрагменты и заранее заданные сущности и связи. Связи `MENTIONS` создаются по совпадению имён и псевдонимов в тексте. Это ручной property graph, а не автоматическое извлечение через `PropertyGraphIndex`.

Используем существующий граф без изменения данных. Если граф ещё не заполнен, выполните из корня репозитория `uv run python llamaindex/advanced_rag_llamaindex_demo.py graph-seed` без `--reset`. В примере используются `NEO4J_URL`, `NEO4J_USERNAME` и `NEO4J_PASSWORD` из окружения; значения по умолчанию соответствуют `docker-compose.yaml`.

In [10]:
from neo4j import GraphDatabase

# Учетные данные по умолчанию берутся из настроек существующего демо.
driver = GraphDatabase.driver(
    demo.NEO4J_URL, auth=(demo.NEO4J_USERNAME, demo.NEO4J_PASSWORD)
)
try:
    driver.verify_connectivity()
    with driver.session() as session:
        counts = session.run("""
            MATCH (n) WHERE n.graph_node = true
            RETURN count(n) AS nodes
        """).single()
    print("Узлов в графе:", counts["nodes"])
    if counts["nodes"] == 0:
        print("Граф пуст. Выполните graph-seed, как указано выше.")
finally:
    driver.close()

Узлов в графе: 105


### Поиск путей по слову

Повторим запрос `graph-search` из проекта: ищем слово в имени или описании конечных узлов и возвращаем пути длиной один или два ребра. Это поиск по подстроке, без семантического ранжирования. В CLI выводится только число путей; здесь дополнительно покажем сами пути.

In [11]:
term = "Atlas"
driver = GraphDatabase.driver(demo.NEO4J_URL, auth=(demo.NEO4J_USERNAME, demo.NEO4J_PASSWORD))
try:
    with driver.session() as session:
        paths = session.run("""
            MATCH p=(n)-[r*1..2]-(m)
            WHERE n.graph_node = true AND m.graph_node = true
              AND (toLower(coalesce(n.name, '') + ' ' + coalesce(n.content, '')) CONTAINS toLower($term)
                OR toLower(coalesce(m.name, '') + ' ' + coalesce(m.content, '')) CONTAINS toLower($term))
            RETURN [node IN nodes(p) | coalesce(node.name, node.id)] AS nodes,
                   [rel IN relationships(p) | type(rel)] AS relationships
            LIMIT 25
        """, term=term).data()
finally:
    driver.close()
print("Найдено путей:", len(paths))
display(pd.DataFrame(paths))

Найдено путей: 25


,nodes,relationships
0,"[Projects, Встреча Atlas и Security по графовому RAG]",[IN_CATEGORY]
1,"[Projects, Встреча Atlas и Security по графовому RAG, data/knowledge_base/Meeting_Atlas_Security_2026.md chunk 33]","[IN_CATEGORY, HAS_CHUNK]"
2,"[Projects, Проект Atlas: графовая навигация по корпоративным знаниям]",[IN_CATEGORY]
3,"[Projects, Проект Atlas: графовая навигация по корпоративным знаниям, data/knowledge_base/Project_Atlas_Graph_2026.md chunk 36]","[IN_CATEGORY, HAS_CHUNK]"
4,"[2026, Встреча Atlas и Security по графовому RAG]",[IN_YEAR]
5,"[2026, Встреча Atlas и Security по графовому RAG, data/knowledge_base/Meeting_Atlas_Security_2026.md chunk 33]","[IN_YEAR, HAS_CHUNK]"
6,"[2026, Проект Atlas: графовая навигация по корпоративным знаниям]",[IN_YEAR]
7,"[2026, Проект Atlas: графовая навигация по корпоративным знаниям, data/knowledge_base/Project_Atlas_Graph_2026.md chunk 36]","[IN_YEAR, HAS_CHUNK]"
8,"[Инструкция по Настройке Integrated Development Environment (IDE) JetBrains для РобоТех, 2026, Встреча Atlas и Security по графовому RAG]","[IN_YEAR, IN_YEAR]"
9,"[Инструкция по Настройке Integrated Development Environment (IDE) JetBrains для РобоТех, 2026, Проект Atlas: графовая навигация по корпоративным знаниям]","[IN_YEAR, IN_YEAR]"


### Два перехода: лидер проекта и используемые системы

Здесь задаём типы связей явно: `Person -[:LEADS]-> Project -[:USES]-> System`. Это пример навигации по существующему графу; он не преобразует произвольный вопрос в Cypher и не генерирует ответ.

In [12]:
driver = GraphDatabase.driver(demo.NEO4J_URL, auth=(demo.NEO4J_USERNAME, demo.NEO4J_PASSWORD))
try:
    with driver.session() as session:
        result = session.run("""
            MATCH (person:Person)-[:LEADS]->(project:Project)-[:USES]->(system:System)
            WHERE person.graph_node = true AND project.graph_node = true AND system.graph_node = true
              AND project.id = $project_id
            RETURN person.name AS leader, project.name AS project, system.name AS system
            ORDER BY system
        """, project_id="project:atlas").data()
finally:
    driver.close()
display(pd.DataFrame(result))

,leader,project,system
0,Иван Петров,Project Atlas,Neo4j
1,Иван Петров,Project Atlas,Qdrant


### Какие фрагменты упоминают проект

Связи `HAS_CHUNK` и `MENTIONS` позволяют перейти от сущности к документам. В Neo4j хранится только сокращённый текст `Chunk`. Ниже заново разбиваем текущие исходные документы тем же способом и показываем полные тексты найденных фрагментов. ID, длина и начало текста сверяются с графом; если они не совпадают, показываем предупреждение. Такая сверка не гарантирует, что текст после сохранённого начала не изменился.


In [13]:
driver = GraphDatabase.driver(demo.NEO4J_URL, auth=(demo.NEO4J_USERNAME, demo.NEO4J_PASSWORD))
try:
    with driver.session() as session:
        sources = session.run("""
            MATCH (doc:Document)-[:HAS_CHUNK]->(chunk:Chunk)-[:MENTIONS]->(project:Project)
            WHERE doc.graph_node = true AND chunk.graph_node = true AND project.graph_node = true
              AND project.id = $project_id
            RETURN doc.source AS source, chunk.id AS chunk_id,
                   chunk.content AS excerpt, chunk.chars AS chars
            ORDER BY source, chunk_id
            LIMIT 10
        """, project_id="project:atlas").data()
finally:
    driver.close()

# Восстанавливаем ID так же, как при создании графа в seed_manual_graph.
graph_splitter = demo.SentenceSplitter(chunk_size=demo.CHUNK_SIZE, chunk_overlap=demo.CHUNK_OVERLAP)
graph_chunks = graph_splitter.get_nodes_from_documents(demo.load_documents())
full_chunks = {
    f"chunk:{chunk.metadata['source']}:{i}": chunk.get_content(metadata_mode="none")
    for i, chunk in enumerate(graph_chunks, 1)
}
for item in sources:
    print(f"\n--- {item['source']} ({item['chunk_id']}) ---")
    full_text = full_chunks.get(item['chunk_id'])
    if (full_text is not None and len(full_text) == item['chars']
            and demo.text_excerpt(full_text, 1100) == item['excerpt']):
        print(full_text)
    else:
        print("Не удалось сопоставить чанк с текущими файлами; показан сокращённый текст из Neo4j:")
        print(item['excerpt'])
if not sources:
    print("Фрагменты не найдены.")



--- data/knowledge_base/Meeting_Atlas_Security_2026.md (chunk:data/knowledge_base/Meeting_Atlas_Security_2026.md:33) ---
# Встреча Atlas и Security по графовому RAG

Дата: 2026-03-14.

Участники: Иван Петров, Мария Соколова, Анна Ким, команда Security, команда AI Platform.

Решения:
- Для графового поиска используется Neo4j PropertyGraphIndex.
- Для точных кодов политик используется hybrid search с sparse-сигналом BM25.
- Для финального порядка источников используется Cross-Encoder Reranking.
- Для вопросов про текущие HR-правила обязателен metadata filter year=2026 и category=HR.

Иван Петров должен подготовить демонстрацию Graph Retrieval для вопроса: "Кто отвечает за проект Atlas и какие системы он использует?"

Мария Соколова должна проверить корректность ответа по политике HR-REMOTE-2026.

--- data/knowledge_base/Project_Atlas_Graph_2026.md (chunk:data/knowledge_base/Project_Atlas_Graph_2026.md:36) ---
# Проект Atlas: графовая навигация по корпоративным знаниям

Проект Atlas стро

Графовый пример показывает связи и фрагменты, но не подключён к генерации ответа выше. Dense/hybrid retrieval и reranking в этом ноутбуке работают через Qdrant.